# PREPROCESSING

- filter relevant node and edge types
- prune low-degree nodes
- re index nodes for GNN compatibility
- construct PyTorch Geometric HeteroData object

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q torch-geometric

In [4]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import torch
from torch_geometric.data import HeteroData

In [6]:
DATA_DIR="/content/drive/MyDrive/drug_repurposing/data"

In [9]:
nodes_path=f"{DATA_DIR}/hetionet-v1.0-nodes.tsv"
nodes_df=pd.read_csv(nodes_path, sep="\t")

In [10]:
edges_path = f"{DATA_DIR}/edges.sif"
edges_df = pd.read_csv(edges_path, sep="\t", header=None, names=["source", "relation", "target"])

In [11]:
edges_df = edges_df.iloc[1:].reset_index(drop=True)

In [12]:
KEEP_NODE_TYPES=["Compound", "Disease", "Gene"]
filtered_nodes=nodes_df[nodes_df["kind"].isin(KEEP_NODE_TYPES)].copy()
filtered_nodes["kind"].value_counts()

,count
kind,
Gene,20945
Compound,1552
Disease,137


In [13]:
id_to_type = dict(zip(nodes_df["id"], nodes_df["kind"]))

def edge_is_valid(row):
    src_type = id_to_type.get(row["source"])
    tgt_type = id_to_type.get(row["target"])

    if src_type not in KEEP_NODE_TYPES or tgt_type not in KEEP_NODE_TYPES:
        return False

    if row["relation"] == "CtD":
        return True

    if src_type == "Compound" and tgt_type == "Gene":
        return True
    if src_type == "Disease" and tgt_type == "Gene":
        return True
    if src_type == "Gene" and tgt_type == "Gene":
        return True

    return False

filtered_edges = edges_df[edges_df.apply(edge_is_valid, axis=1)].copy()

filtered_edges["relation"].value_counts()

,count
relation,
Gr>G,265672
GiG,147164
GcG,61690
CdG,21102
CuG,18756
DaG,12623
CbG,11571
DuG,7731
DdG,7623


In [14]:
ctd_edges = filtered_edges[filtered_edges["relation"] == "CtD"]
print("total CtD edges:", len(ctd_edges))

total CtD edges: 755


In [16]:
#remove noisy singleton

MIN_DEGREE = 2

drug_degree = Counter(ctd_edges["source"])
disease_degree = Counter(ctd_edges["target"])

valid_drugs = {d for d, deg in drug_degree.items() if deg >= MIN_DEGREE}
valid_diseases = {d for d, deg in disease_degree.items() if deg >= MIN_DEGREE}

filtered_edges = filtered_edges[(~filtered_edges["relation"].eq("CtD")) | (filtered_edges["source"].isin(valid_drugs) & filtered_edges["target"].isin(valid_diseases))]

In [17]:
used_node_ids=set(filtered_edges["source"]) | set(filtered_edges["target"])

filtered_nodes=filtered_nodes[
    filtered_nodes["id"].isin(used_node_ids)
].copy()

filtered_nodes["kind"].value_counts()

,count
kind,
Gene,18270
Compound,1441
Disease,134


In [18]:
node_maps={}
reverse_node_maps={}

for node_type in KEEP_NODE_TYPES:
    ids=filtered_nodes[filtered_nodes["kind"] == node_type]["id"].tolist()
    node_maps[node_type]={nid: i for i, nid in enumerate(ids)}
    reverse_node_maps[node_type]={i: nid for nid, i in node_maps[node_type].items()}

In [19]:
data = HeteroData()

for node_type in KEEP_NODE_TYPES:
    num_nodes = len(node_maps[node_type])
    data[node_type].num_nodes = num_nodes

In [20]:
def add_edges(src_type, rel, tgt_type):
    subset=filtered_edges[
        (filtered_edges["relation"] == rel) &
        (filtered_edges["source"].map(id_to_type) == src_type) &
        (filtered_edges["target"].map(id_to_type) == tgt_type)
    ]

    if len(subset)==0:
        return

    src_idx=subset["source"].map(node_maps[src_type]).values
    tgt_idx=subset["target"].map(node_maps[tgt_type]).values

    edge_index=torch.tensor([src_idx, tgt_idx], dtype=torch.long)
    data[(src_type, rel, tgt_type)].edge_index=edge_index

In [21]:
add_edges("Compound", "CtD", "Disease")

#populate relations
for rel in filtered_edges["relation"].unique():
    add_edges("Compound", rel, "Gene")
    add_edges("Disease", rel, "Gene")
    add_edges("Gene", rel, "Gene")

/tmp/ipython-input-4261285626.py:14: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index=torch.tensor([src_idx, tgt_idx], dtype=torch.long)


In [22]:
print(data)

HeteroData(
  Compound={ num_nodes=1441 },
  Disease={ num_nodes=134 },
  Gene={ num_nodes=18270 },
  (Compound, CtD, Disease)={ edge_index=[2, 487] },
  (Gene, GiG, Gene)={ edge_index=[2, 147164] },
  (Disease, DdG, Gene)={ edge_index=[2, 7623] },
  (Compound, CbG, Gene)={ edge_index=[2, 11571] },
  (Compound, CuG, Gene)={ edge_index=[2, 18756] },
  (Disease, DaG, Gene)={ edge_index=[2, 12623] },
  (Gene, GcG, Gene)={ edge_index=[2, 61690] },
  (Gene, Gr>G, Gene)={ edge_index=[2, 265672] },
  (Compound, CdG, Gene)={ edge_index=[2, 21102] },
  (Disease, DuG, Gene)={ edge_index=[2, 7731] }
)


In [26]:
import os

OUTPUT_PATH = f"{DATA_DIR}/processed/filtered_graph.pt"

# Create parent directory if it doesn't exist
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

torch.save(
    {
        "data": data,
        "node_maps": node_maps,
        "reverse_node_maps": reverse_node_maps
    },
    OUTPUT_PATH
)
print("saved processed graph to:", OUTPUT_PATH)

saved processed graph to: /content/drive/MyDrive/drug_repurposing/data/processed/filtered_graph.pt
